## Notebook 07: Validation Against SOCA

This notebook validates SENSE results against the SOCA database (Social Hotspots Database). It compares social risk values computed by SENSE (using PSILCA) with reference values from SOCA EcoSpold01 files.

**What it does:**
1. Parses SOCA EcoSpold01 XML files and links them to ecoinvent activities via dataset IDs.
2. Extracts social flows from the Brightway database (SENSE/PSILCA side).
3. Computes log10(SENSE / SOCA) per (activity, indicator) pair.
4. Produces a three-panel histogram stratified by geography class (all locations, countries and states, regions and other areas).
5. Exports an Excel file listing activities present in only one of the two systems.

**Inputs:**
- Brightway projects `sense_validation_arithmetic` and `sense_validation_geometric` from notebooks 00 and 01.
- SOCA EcoSpold01 export (`Data/soca/EcoSpold01/`).
- Ecoinvent overview Excel with "Cut-Off AO" and "Geographies" sheets.

**Outputs:**
- `validation_arithmetic.svg` and `validation_geometric.svg`: three-panel histograms.
- `sense_soca_activity_coverage.xlsx`: activities only present in one system.

In [ ]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd
from bw2data import projects, Database

try:
    from tqdm.auto import tqdm
except ImportError:  # optional dependency
    def tqdm(iterable, **kwargs):
        return iterable

import matplotlib.pyplot as plt


# ============================================================
# USER CONFIG – ONLY THESE 4 NEED TO BE ADAPTED
# ============================================================
PROJECT_NAME      = "sense_validation_arithmetic" # or "sense_validation_geometric"
TECHNOSPHERE_DB   = "ecoinvent-3.11-cutoff"

SPOLD_FOLDER      = Path(r"../Data/soca/EcoSpold01")
SPOLD_PATTERN     = "*.xml"  # usually '*.xml' or '*.spold'
OVERVIEW_XLSX     = Path(r"../Data/Database-Overview-for-ecoinvent-v3.11.xlsx")

# Where to write the Excel file listing activities only present in one DB
COVERAGE_EXCEL    = Path("sense_soca_activity_coverage.xlsx")

# ============================================================
# CACHING (OPTION 1): Parse all XML once, then load cached data
# ============================================================
USE_CACHE = True
FORCE_REBUILD_CACHE = False  # set True to ignore cache and re-parse once

# Cache location: next to this script if possible, otherwise cwd
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

CACHE_DIR = BASE_DIR / ".cache_soca"
CACHE_DIR.mkdir(exist_ok=True)

CACHE_MAPPING = CACHE_DIR / "mapping_df.pkl"
CACHE_XML     = CACHE_DIR / "xml_df.pkl"
CACHE_META    = CACHE_DIR / "spold_signature.txt"


# ============================================================
# CONSTANTS
# ============================================================
NS1 = {"spold": "http://www.EcoInvent.org/EcoSpold01"}

STAKEHOLDER_GROUPS = {
    "Society",
    "Workers",
    "Local Community",
    "Value Chain Actors",
    "Children",
    "Consumer",
}

LEVEL_WEIGHTS = {
    "no": 0.001,
    "very low": 0.01,
    "low": 0.1,
    "medium": 1,
    "high": 10,
    "very high": 100,
    "no data": 0.1,  # convention: treat "no data" like "medium"
}


# ============================================================
# SMALL HELPERS
# ============================================================
def norm(x):
    """Normalize strings: strip, lower; keep None as None."""
    return None if x is None else str(x).strip().lower()


def level_to_weight(level):
    """Map a level string to its numerical weight (case-insensitive)."""
    if not isinstance(level, str):
        return None
    return LEVEL_WEIGHTS.get(level.strip().lower())


def get_stakeholder_from_categories(categories):
    """
    From a Brightway 'categories' field, extract:
      - stakeholder_group (e.g. 'Society')
      - subgroup         (e.g. 'Technology development')
    for strings of the form 'Stakeholder - Subgroup'.
    """
    if not categories:
        return None, None

    if isinstance(categories, (list, tuple)):
        cat_elems = [str(c) for c in categories]
    else:
        cat_elems = [str(categories)]

    candidate = None
    for c in cat_elems:
        for sh in STAKEHOLDER_GROUPS:
            if c.startswith(sh):  # e.g. "Society - Technology development"
                candidate = c
                break
        if candidate is not None:
            break

    if candidate is None:
        return None, None

    if " - " in candidate:
        stakeholder_group, subgroup = candidate.split(" - ", 1)
        stakeholder_group = stakeholder_group.strip()
        subgroup = subgroup.strip()
    else:
        stakeholder_group = candidate.strip()
        subgroup = None

    if stakeholder_group not in STAKEHOLDER_GROUPS:
        return None, None

    return stakeholder_group, subgroup


def clean_indicator_name(flow_name):
    """
    Clean sense flow names such as:
        'Ecosystem services (risk), RoW'
        'Extraction of biomass (related to area) (risk), RoW'

    Keep everything up to the *last* ' ('.
    """
    if flow_name is None:
        return None
    s = str(flow_name).strip()
    idx = s.rfind(" (")
    return s if idx == -1 else s[:idx].rstrip()


def parse_spold_name_field(raw_name):
    """
    Parse the XML 'name' field into:
      - indicator: text before the first ';'
      - level: text after '; ' and before the last token
      - type: last token, expected to be 'risk' or 'opportunity'

    Example:
      'Internet freedom scores; very low risk'
        -> indicator='Internet freedom scores'
           level='very low'
           type='risk'

    Special case:
      if the part after ';' is exactly 'no data' (case-insensitive),
      then:
        level='no data'
        type=None
    """
    if raw_name is None:
        return None, None, None

    txt = str(raw_name).strip()
    if not txt:
        return None, None, None

    parts = txt.split(";", 1)
    indicator = parts[0].strip()
    rest = parts[1].strip() if len(parts) > 1 else ""

    if not rest:
        return indicator or None, None, None

    if rest.lower() == "no data":
        return indicator or None, "no data", None

    tokens = rest.split()
    if not tokens:
        return indicator or None, None, None

    last = tokens[-1].lower()
    if last in {"risk", "opportunity"}:
        type_ = last
        level = " ".join(tokens[:-1]).strip() or None
    else:
        type_ = None
        level = rest or None

    return indicator or None, level, type_


def read_excel_sheet(path: Path, sheet_name: str, **read_kwargs):
    """
    Wrapper around pandas.read_excel with friendlier error messages.
    """
    try:
        return pd.read_excel(path, sheet_name=sheet_name, **read_kwargs)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Excel file not found: {path}\n"
            f"Please set OVERVIEW_XLSX correctly in the USER CONFIG section."
        )
    except ValueError as e:
        # Typically "Worksheet named 'XYZ' not found"
        raise ValueError(
            f"Sheet '{sheet_name}' not found in {path}.\n"
            f"Please make sure you're using the ecoinvent overview file "
            f"with the expected sheet names ('Cut-Off AO', 'Geographies')."
        ) from e


# ============================================================
# CACHE HELPERS
# ============================================================
def spold_signature(folder: Path, pattern: str) -> str:
    """
    Simple signature of the EcoSpold folder:
      - number of matching files
      - newest modification time among them

    If you change/add files, signature changes -> cache invalidates.
    """
    files = sorted(folder.glob(pattern))
    if not files:
        return "EMPTY"
    newest = max(f.stat().st_mtime for f in files)
    return f"{len(files)}|{newest}"


def load_or_parse_spold():
    """
    Either load mapping_df/xml_df from cache, or parse all files and cache them.
    """
    sig = spold_signature(SPOLD_FOLDER, SPOLD_PATTERN)

    if USE_CACHE and not FORCE_REBUILD_CACHE:
        if CACHE_MAPPING.exists() and CACHE_XML.exists() and CACHE_META.exists():
            old_sig = CACHE_META.read_text().strip()
            if old_sig == sig:
                print(f"Loading cached EcoSpold parses from: {CACHE_DIR}")
                mapping_df = pd.read_pickle(CACHE_MAPPING)
                xml_df = pd.read_pickle(CACHE_XML)
                print(f"  Cached mapping_df rows: {len(mapping_df)}")
                print(f"  Cached xml_df rows:     {len(xml_df)}\n")
                return mapping_df, xml_df

    # Parse and cache
    mapping_df, xml_df = process_spold_folder()

    if USE_CACHE:
        print(f"Saving EcoSpold cache to: {CACHE_DIR}")
        mapping_df.to_pickle(CACHE_MAPPING)
        xml_df.to_pickle(CACHE_XML)
        CACHE_META.write_text(sig)
        print("  Cache saved.\n")

    return mapping_df, xml_df


# ============================================================
# STEP 1 – Parse EcoSpold once → metadata + SOCA social flows
# ============================================================
def parse_spold_file(path: Path):
    """
    Parse one EcoSpold01 XML file and return:
      - metadata row (for mapping_df)
      - list of social-flow rows (for xml_df)

    Social flows are only recorded if a dataset_id was found.
    """
    text = path.read_text(encoding="utf-8", errors="ignore")

    # ecoQuery URL → (version, system_model, dataset_id)
    m = re.search(
        r"https://ecoquery\.ecoinvent\.org/([^/]+)/([^/]+)/dataset/(\d+)/",
        text,
    )
    if m:
        version, system_model, dataset_id = m.groups()
        dataset_id = int(dataset_id)
    else:
        version = system_model = None
        dataset_id = None

    root = ET.fromstring(text)
    rf = root.find(".//spold:referenceFunction", NS1)
    geo = root.find(".//spold:geography", NS1)

    product = rf.get("name") if rf is not None else None
    unit = rf.get("unit") if rf is not None else None
    location = geo.get("location") if geo is not None else None

    meta_row = {
        "file": path,
        "file_name": path.name,
        "dataset_id": dataset_id,
        "eco_version": version,
        "system_model": system_model,
        "product": product,
        "unit": unit,
        "location": location,
    }

    social_rows = []
    if dataset_id is not None:
        for exc in root.findall(".//spold:exchange", NS1):
            cat = exc.get("category")
            if cat not in STAKEHOLDER_GROUPS:
                continue

            subcat = exc.get("subCategory")
            raw_name = exc.get("name")
            exc_unit = exc.get("unit")
            mean_val_str = exc.get("meanValue")

            try:
                mean_value = float(mean_val_str) if mean_val_str is not None else None
            except ValueError:
                mean_value = None

            indicator, level, type_ = parse_spold_name_field(raw_name)
            w = level_to_weight(level)
            medium_hours = (
                mean_value * w if (mean_value is not None and w is not None) else None
            )

            social_rows.append(
                {
                    "dataset_id": dataset_id,
                    "file_name": path.name,
                    "eco_version": version,
                    "system_model": system_model,
                    "product": product,
                    "location": location,
                    "stakeholder_group": cat,
                    "subgroup": subcat,
                    "indicator": indicator,
                    "level": level,
                    "type": type_,
                    "unit": exc_unit,
                    "mean_value": mean_value,
                    "level_weight": w,
                    "medium_risk_opportunity_hours": medium_hours,
                    "raw_name": raw_name,
                    "exchange_number": exc.get("number"),
                }
            )

    return meta_row, social_rows


def process_spold_folder():
    """
    Parse all SOCA EcoSpold01 files into:
      - mapping_df : per-file metadata with dataset_id
      - xml_df     : social flows extracted from XML
    """
    print("Parsing EcoSpold01 files...")
    spold_files = sorted(SPOLD_FOLDER.glob(SPOLD_PATTERN))
    if not spold_files:
        raise FileNotFoundError(
            f"No files matching {SPOLD_PATTERN!r} in {SPOLD_FOLDER.resolve()}\n"
            "Please set SPOLD_FOLDER and SPOLD_PATTERN correctly in the USER CONFIG section."
        )

    meta_rows = []
    social_rows = []

    for path in tqdm(spold_files, desc="EcoSpold files"):
        meta_row, s_rows = parse_spold_file(path)
        meta_rows.append(meta_row)
        social_rows.extend(s_rows)

    mapping_df = pd.DataFrame(meta_rows)
    mapping_df = mapping_df.dropna(subset=["dataset_id"]).copy()
    mapping_df["dataset_id"] = mapping_df["dataset_id"].astype("int64")

    xml_df = pd.DataFrame(social_rows)
    if not xml_df.empty:
        xml_df["dataset_id"] = xml_df["dataset_id"].astype("Int64")

    print(f"  EcoSpold files with dataset_id: {len(mapping_df)}")
    print(f"  Unique dataset_ids in XML:      {mapping_df['dataset_id'].nunique()}")
    print(f"  Social exchanges from XML:      {len(xml_df)}\n")

    return mapping_df, xml_df


# ============================================================
# STEP 2 – AO merge + Brightway matching
# ============================================================
def build_ao_and_bw_match(mapping_df, eco_db):
    """
    Merge EcoSpold metadata with the AO Excel and match AO activities
    to Brightway activities via (name, ref product, geography).

    Returns spold_with_ao (includes bw_key_meta_no_unit).
    """
    print("Reading overview Excel (Cut-Off AO)...")
    cutoff = read_excel_sheet(
        OVERVIEW_XLSX,
        "Cut-Off AO",
        keep_default_na=False,  # keep 'NA' as geography, not NaN
    )

    cutoff["dataset_id"] = (
        cutoff["ecoQuery URL"].str.extract(r"/dataset/(\d+)/").astype("Int64")
    )

    cutoff_small = cutoff[
        [
            "dataset_id",
            "Activity UUID",
            "Activity Name",
            "Geography",
            "Reference Product Name",
            "Unit",
        ]
    ].dropna(subset=["dataset_id"]).copy()

    cutoff_small["dataset_id"] = cutoff_small["dataset_id"].astype("int64")

    print(f"  AO rows with dataset_id: {len(cutoff_small)}")
    print(f"  AO unique dataset_ids:  {cutoff_small['dataset_id'].nunique()}")

    spold_with_ao = mapping_df.merge(cutoff_small, on="dataset_id", how="left")
    print(f"  Merged rows (spold_with_ao): {len(spold_with_ao)}")

    # ---- Build Brightway index on (name, reference product, location)
    print("Building Brightway index (name, ref product, location)...")
    bw_index = defaultdict(list)
    for act in eco_db:
        name = act.get("name")
        ref_product = act.get("reference product")
        loc = act.get("location")
        if not (name and ref_product and loc):
            continue
        key = (norm(name), norm(ref_product), norm(loc))
        bw_index[key].append(act)
    print(f"  Brightway index size: {len(bw_index)}")

    # ---- Match per unique metadata combination
    meta_cols = ["Activity Name", "Reference Product Name", "Geography"]
    unique_meta = spold_with_ao[meta_cols].drop_duplicates().reset_index(drop=True)

    def match_bw(row):
        key = (
            norm(row["Activity Name"]),
            norm(row["Reference Product Name"]),
            norm(row["Geography"]),
        )
        candidates = bw_index.get(key, [])
        if len(candidates) == 1:
            act = candidates[0]
            return pd.Series(
                {
                    "bw_key_meta_no_unit": act.key,
                    "bw_name_meta_no_unit": act["name"],
                    "meta_no_unit_n_candidates": 1,
                }
            )
        else:
            return pd.Series(
                {
                    "bw_key_meta_no_unit": None,
                    "bw_name_meta_no_unit": None,
                    "meta_no_unit_n_candidates": len(candidates),
                }
            )

    uniq_match = unique_meta.apply(match_bw, axis=1)
    unique_meta_matched = pd.concat([unique_meta, uniq_match], axis=1)

    spold_with_ao = spold_with_ao.merge(
        unique_meta_matched, on=meta_cols, how="left"
    )

    n_matched = spold_with_ao["bw_key_meta_no_unit"].notna().sum()
    print(f"  Brightway activities uniquely matched: {n_matched}\n")

    return spold_with_ao


# ============================================================
# STEP 3 – Extract social flows from Brightway (SENSE)
# ============================================================
def extract_bw_social_flows(eco_db, spold_with_ao):
    """
    For all activities successfully matched to AO (bw_key_meta_no_unit),
    collect all biosphere exchanges that belong to SENSE stakeholder compartments,
    and return them as a DataFrame.
    """
    target_keys = set(spold_with_ao["bw_key_meta_no_unit"].dropna())
    print(f"Extracting social flows from Brightway for {len(target_keys)} activities...")

    eco_data = eco_db.load()  # dict: {(db_name, code): dataset_dict}

    biosphere_data_cache = {}  # {biosphere_db_name: bio_db.load()}
    flow_meta_cache = {}       # {(db, code): meta_dict or None}
    sentinel = object()

    rows = []

    for act_key, ds in tqdm(
        eco_data.items(),
        total=len(eco_data),
        desc="Scanning technosphere activities",
    ):
        if act_key not in target_keys:
            continue

        act_name = ds.get("name")
        act_loc = ds.get("location")
        act_rp = ds.get("reference product")

        for exc in ds.get("exchanges", []):
            if exc.get("type") != "biosphere":
                continue

            flow_db, flow_code = exc["input"]
            flow_key = (flow_db, flow_code)

            meta = flow_meta_cache.get(flow_key, sentinel)
            if meta is sentinel:
                # First time we see this flow → fetch biosphere metadata
                bio_data = biosphere_data_cache.get(flow_db)
                if bio_data is None:
                    bio_data = Database(flow_db).load()
                    biosphere_data_cache[flow_db] = bio_data

                obj = bio_data.get(flow_key, {})
                stakeholder_group, subgroup = get_stakeholder_from_categories(
                    obj.get("categories")
                )
                if stakeholder_group is None:
                    flow_meta_cache[flow_key] = None
                    continue

                raw_flow_name = obj.get("name")
                meta = {
                    "flow_name_raw": raw_flow_name,
                    "indicator": clean_indicator_name(raw_flow_name),
                    "flow_unit": obj.get("unit"),
                    "stakeholder_group": stakeholder_group,
                    "subgroup": subgroup,
                    "raw_categories": obj.get("categories"),
                }
                flow_meta_cache[flow_key] = meta

            if meta is None:
                continue  # not a social flow of interest

            rows.append(
                {
                    "activity_key": act_key,
                    "activity_name": act_name,
                    "activity_location": act_loc,
                    "activity_ref_product": act_rp,
                    "stakeholder_group": meta["stakeholder_group"],
                    "subgroup": meta["subgroup"],
                    "indicator": meta["indicator"],
                    "unit": meta["flow_unit"],
                    "amount": exc["amount"],
                    "flow_key": flow_key,
                    "flow_name_raw": meta["flow_name_raw"],
                    "raw_categories": meta["raw_categories"],
                }
            )

    bw_df = pd.DataFrame(rows)
    print(f"  Social exchanges extracted from Brightway: {len(bw_df)}")

    # Link back to dataset_id via spold_with_ao
    mapping = (
        spold_with_ao[["bw_key_meta_no_unit", "dataset_id"]]
        .dropna(subset=["bw_key_meta_no_unit"])
        .drop_duplicates()
        .rename(columns={"bw_key_meta_no_unit": "activity_key"})
    )

    bw_df = bw_df.merge(mapping, on="activity_key", how="left")
    bw_df["dataset_id"] = bw_df["dataset_id"].astype("Int64")

    n_with_id = bw_df["dataset_id"].notna().sum()
    print(f"  Social exchanges with dataset_id: {n_with_id}\n")

    return bw_df


# ============================================================
# STEP 4 – Indicator overview (quick sanity check)
# ============================================================
def summarize_indicators(bw_df, xml_df):
    """Print simple counts of indicators present in each system."""
    bw_indicators = set(bw_df["indicator"].dropna())
    xml_indicators = set(xml_df["indicator"].dropna())

    print("Indicator overview")
    print("------------------")
    print(f"  Unique indicators in SENSE / Brightway: {len(bw_indicators)}")
    print(f"  Unique indicators in SOCA / XML:        {len(xml_indicators)}")
    print(f"  Indicators present in both:             {len(bw_indicators & xml_indicators)}")
    print(f"  Only in SENSE / Brightway:              {len(bw_indicators - xml_indicators)}")
    print(f"  Only in SOCA / XML:                     {len(xml_indicators - bw_indicators)}\n")


# ============================================================
# STEP 5 – Build compare_pos (dataset_id, indicator)
# ============================================================
def build_compare(bw_df, xml_df):
    """
    Build compare_pos with:
      - dataset_id, indicator
      - sense_value (SENSE/Brightway)
      - soca_value  (SOCA/XML)
      - log_diff    = log10(SENSE / SOCA)
    keeping only strictly positive pairs.
    """
    print("Preparing data for comparison...")

    bw_df = bw_df.dropna(subset=["dataset_id", "indicator"]).copy()
    xml_df = xml_df.dropna(
        subset=["dataset_id", "indicator", "medium_risk_opportunity_hours"]
    ).copy()

    bw_df["dataset_id"] = bw_df["dataset_id"].astype("int64")
    xml_df["dataset_id"] = xml_df["dataset_id"].astype("int64")

    bw_ids = set(bw_df["dataset_id"].unique())
    xml_ids = set(xml_df["dataset_id"].unique())
    both_ids = bw_ids & xml_ids

    print(f"  Activities with social data in BOTH systems: {len(both_ids)}")

    bw_df = bw_df[bw_df["dataset_id"].isin(both_ids)].copy()
    xml_df = xml_df[xml_df["dataset_id"].isin(both_ids)].copy()

    # Aggregate per (dataset_id, indicator)
    bw_agg = (
        bw_df.groupby(["dataset_id", "indicator"], dropna=False)["amount"]
        .sum()
        .reset_index()
        .rename(columns={"amount": "sense_value"})
    )

    xml_agg = (
        xml_df.groupby(["dataset_id", "indicator"], dropna=False)[
            "medium_risk_opportunity_hours"
        ]
        .sum()
        .reset_index()
        .rename(columns={"medium_risk_opportunity_hours": "soca_value"})
    )

    compare = bw_agg.merge(xml_agg, on=["dataset_id", "indicator"], how="inner")
    print(
        f"  (dataset_id, indicator) pairs present in BOTH systems: {len(compare)}"
    )

    # Keep only strictly positive values for log-ratio
    mask_pos = (compare["sense_value"] > 0) & (compare["soca_value"] > 0)
    compare_pos = compare[mask_pos].copy()

    compare_pos["log_diff"] = (
        np.log10(compare_pos["sense_value"]) - np.log10(compare_pos["soca_value"])
    )

    print(
        f"  Pairs with positive values in BOTH systems: {len(compare_pos)}"
    )

    print("\nSummary of log10(SENSE / SOCA) (SOCA as reference):")
    print(compare_pos["log_diff"].describe())

    typical_factor = 10 ** compare_pos["log_diff"].abs().mean()
    print(f"\nTypical absolute factor difference ≈ {typical_factor:.2f}x\n")

    return compare_pos


# ============================================================
# STEP 6 – Activity-level coverage & Excel export
# ============================================================
def summarize_activity_coverage_and_export(bw_df, xml_df, spold_with_ao, output_path: Path):
    """
    Summarize coverage at activity (dataset_id) level and write an Excel file
    listing activities that have social flows only in one of the systems.

    "Present" means: activity has at least one social flow in that system.
    """
    # Set of all activities in AO/spold universe
    all_ids = set(spold_with_ao["dataset_id"].dropna().astype("int64"))

    # Activities with social flows in each system
    bw_ids = set(bw_df["dataset_id"].dropna().astype("int64"))
    xml_ids = set(xml_df["dataset_id"].dropna().astype("int64"))

    both_included   = bw_ids & xml_ids
    only_sense     = bw_ids - xml_ids
    only_soca       = xml_ids - bw_ids
    excluded_both   = all_ids - (bw_ids | xml_ids)

    print("Activity-level coverage (dataset_id):")
    print("-------------------------------------")
    print(f"  Total activities in AO/spold universe:     {len(all_ids)}")
    print(f"  Included in BOTH sense and SOCA:          {len(both_included)}")
    print(f"  Only sense (SENSE/Brightway):             {len(only_sense)}")
    print(f"  Only SOCA (EcoSpold XML):                  {len(only_soca)}")
    print(f"  Excluded in BOTH (no social flows):        {len(excluded_both)}\n")

    # Build metadata table and filter
    meta_cols = ["dataset_id", "Activity Name", "Geography",
                 "Reference Product Name", "Unit"]
    meta = spold_with_ao[meta_cols].drop_duplicates()

    only_sense_df = (
        meta[meta["dataset_id"].isin(only_sense)]
        .copy()
        .sort_values(["Activity Name", "Reference Product Name", "Geography"])
    )
    only_soca_df = (
        meta[meta["dataset_id"].isin(only_soca)]
        .copy()
        .sort_values(["Activity Name", "Reference Product Name", "Geography"])
    )

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        only_sense_df.to_excel(writer, sheet_name="Only_SENSE", index=False)
        only_soca_df.to_excel(writer, sheet_name="Only_SOCA", index=False)

    print(f"Saved lists of activities only present in one DB to: {output_path}\n")


# ============================================================
# STEP 7 – Multi-panel geography plot
# ============================================================
def plot_geography_panels(compare_pos, spold_with_ao, savepath=None):
    """
    Reproduce the 3-panel figure:
      - All locations
      - Countries & states
      - Regions & other areas
    using log_diff = log10(SENSE / SOCA).
    """
    # =====================================================
    # CONFIG – x-limits & figure size
    # =====================================================
    XMIN = -4
    XMAX = 4

    ORIG_W_IN, ORIG_H_IN = 8, 8
    MAX_W_MM, MAX_H_MM = 174, 234

    MAX_W_IN = MAX_W_MM / 25.4
    MAX_H_IN = MAX_H_MM / 25.4

    scale = min(MAX_W_IN / ORIG_W_IN, MAX_H_IN / ORIG_H_IN)
    FIG_WIDTH_IN = ORIG_W_IN * scale
    FIG_HEIGHT_IN = ORIG_H_IN * scale

    print(f"Figure size (inches): {FIG_WIDTH_IN:.2f} x {FIG_HEIGHT_IN:.2f}")
    print(
        f"Figure size (mm):     {FIG_WIDTH_IN*25.4:.1f} x {FIG_HEIGHT_IN*25.4:.1f}"
    )

    # =====================================================
    # FONT SIZES & DPI (tuned for 84 mm width)
    # =====================================================
    _scale = 174.0 / 84.0  # scale all visual params for 174 mm width
    SMALL_SIZE = 5 * _scale    # tick labels, annotations
    MEDIUM_SIZE = 6 * _scale   # axis labels
    BIGGER_SIZE = 6.5 * _scale # titles

    plt.rcParams.update(
        {
            "figure.dpi": 300,
            "savefig.dpi": 600,
            "font.size": SMALL_SIZE,
            "axes.titlesize": BIGGER_SIZE,
            "axes.labelsize": MEDIUM_SIZE,
            "xtick.labelsize": SMALL_SIZE,
            "ytick.labelsize": SMALL_SIZE,
            "legend.fontsize": SMALL_SIZE,
            "figure.titlesize": BIGGER_SIZE,
        }
    )

    # =====================================================
    # 1) Attach geography to compare_pos via spold_with_ao
    # =====================================================
    ds_geo = (
        spold_with_ao[["dataset_id", "Geography"]]
        .dropna(subset=["dataset_id", "Geography"])
        .drop_duplicates()
        .copy()
    )
    ds_geo["dataset_id"] = ds_geo["dataset_id"].astype("int64")
    ds_geo["Geography"] = ds_geo["Geography"].astype(str).str.strip()

    print(
        f"dataset_id → Geography mappings: {len(ds_geo)} rows, "
        f"{ds_geo['dataset_id'].nunique()} unique dataset_ids"
    )

    compare_pos_plot = compare_pos.copy()
    compare_pos_plot["dataset_id"] = compare_pos_plot["dataset_id"].astype("int64")

    geo_cols_to_drop = [
        c for c in compare_pos_plot.columns if c.startswith("Geography")
    ]
    if geo_cols_to_drop:
        print(
            "Dropping existing geography columns from compare_pos:",
            geo_cols_to_drop,
        )
        compare_pos_plot = compare_pos_plot.drop(columns=geo_cols_to_drop)

    compare_pos_plot = compare_pos_plot.merge(ds_geo, on="dataset_id", how="left")

    # =====================================================
    # 2) Load Geographies sheet and classify shortnames
    # =====================================================
    geo_df = read_excel_sheet(OVERVIEW_XLSX, "Geographies", dtype=str)
    geo_df["Shortname"] = geo_df["Shortname"].astype(str).str.strip()
    geo_df["Geography Classification"] = (
        geo_df["Geography Classification"].astype(str).str.strip()
    )

    country_state_shortnames = set(
        geo_df.loc[
            geo_df["Geography Classification"].isin(
                ["Country", "Province or State"]
            ),
            "Shortname",
        ].dropna()
    )

    print(
        f"Shortnames classified as Country/State: {len(country_state_shortnames)}"
    )

    compare_pos_plot["Geography"] = (
        compare_pos_plot["Geography"].astype(str).str.strip()
    )

    compare_pos_plot["geo_class"] = np.where(
        compare_pos_plot["Geography"].isin(country_state_shortnames),
        "Country/State",
        "Region/Other",
    )

    all_act_ids = set(compare_pos_plot["dataset_id"].dropna().astype("int64"))
    country_act_ids = set(
        compare_pos_plot.loc[
            compare_pos_plot["geo_class"] == "Country/State", "dataset_id"
        ]
        .dropna()
        .astype("int64")
    )
    region_act_ids = set(
        compare_pos_plot.loc[
            compare_pos_plot["geo_class"] == "Region/Other", "dataset_id"
        ]
        .dropna()
        .astype("int64")
    )

    n_all_acts = len(all_act_ids)
    n_country_acts = len(country_act_ids)
    n_region_acts = len(region_act_ids)

    if n_all_acts > 0:
        pct_country = 100 * n_country_acts / n_all_acts
        pct_region = 100 * n_region_acts / n_all_acts
    else:
        pct_country = pct_region = 0.0

    print(
        f"Total activities with social data in BOTH DBs: {n_all_acts} (100.0%)"
    )
    print(
        f"Activities in countries/states:               {n_country_acts} ({pct_country:.1f}%)"
    )
    print(
        f"Activities in regions/other:                  {n_region_acts} ({pct_region:.1f}%)"
    )

    # =====================================================
    # 3) Pair-level values for plotting (apply x-limits)
    # =====================================================
    def in_range(arr, xmin, xmax):
        arr = np.asarray(arr)
        return arr[(arr >= xmin) & (arr <= xmax)]

    all_vals_raw = compare_pos_plot["log_diff"].dropna().values
    country_vals_raw = compare_pos_plot.loc[
        compare_pos_plot["dataset_id"].isin(country_act_ids), "log_diff"
    ].dropna().values
    region_vals_raw = compare_pos_plot.loc[
        compare_pos_plot["dataset_id"].isin(region_act_ids), "log_diff"
    ].dropna().values

    all_vals_plot = in_range(all_vals_raw, XMIN, XMAX)
    country_vals_plot = in_range(country_vals_raw, XMIN, XMAX)
    region_vals_plot = in_range(region_vals_raw, XMIN, XMAX)

    print(
        f"Pairs used within [{XMIN}, {XMAX}] for ALL:      {len(all_vals_plot)}"
    )
    print(
        f"Pairs used within [{XMIN}, {XMAX}] for COUNTRY:  {len(country_vals_plot)}"
    )
    print(
        f"Pairs used within [{XMIN}, {XMAX}] for REGION:   {len(region_vals_plot)}"
    )

    if len(all_vals_plot) > 0:
        bins = np.linspace(XMIN, XMAX, 40)
    else:
        bins = 50

    groups = [
        ("All locations", all_vals_plot, n_all_acts),
        ("Countries & states", country_vals_plot, n_country_acts),
        ("Regions & other areas", region_vals_plot, n_region_acts),
    ]

    # =====================================================
    # 4) Helper: per-panel y-limit based on its own density
    # =====================================================
    def panel_ymax(vals_plot, bins_):
        vals_plot = np.asarray(vals_plot)
        if vals_plot.size == 0:
            return 1.0
        hist, _ = np.histogram(vals_plot, bins=bins_, density=True)
        if hist.size == 0 or np.all(~np.isfinite(hist)):
            return 1.0
        return float(np.nanmax(hist)) * 1.05

    # =====================================================
    # 5) Plot small multiples (each with its own y-scale)
    # =====================================================
    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        sharex=True,
        sharey=False,
        figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
        constrained_layout=False,
    )

    for ax, (title, vals_plot, n_acts) in zip(axes, groups):
        if len(vals_plot) == 0:
            ax.text(
                0.5,
                0.5,
                "No data",
                ha="center",
                va="center",
                fontsize=SMALL_SIZE,
            )
            ax.set_ylabel("Probability density", fontsize=MEDIUM_SIZE)
            ax.set_title(title, fontsize=BIGGER_SIZE)
            ax.set_ylim(0, 1.0)
            ax.tick_params(axis="both", labelsize=SMALL_SIZE)
            ax.grid(axis="y", linestyle=":", linewidth=0.3 * _scale, alpha=0.5)
            continue

        ymax_panel = panel_ymax(vals_plot, bins)

        ax.hist(
            vals_plot,
            bins=bins,
            density=True,
            alpha=0.7,
            edgecolor="black",
            linewidth=0.3 * _scale,
            histtype="bar",
        )

        ax.grid(axis="y", linestyle=":", linewidth=0.3 * _scale, alpha=0.5)
        ax.axvline(0, linestyle="-", linewidth=0.8 * _scale, color="black")

        median_val = np.median(vals_plot)
        pct_acts = 100 * n_acts / n_all_acts if n_all_acts > 0 else 0.0

        ax.text(
            0.02,
            0.82,
            (
                f"N activities = {n_acts} ({pct_acts:.1f}%)\n"
                f"median log₁₀(SENSE/SOCA) = {median_val:.2f}"
            ),
            transform=ax.transAxes,
            fontsize=SMALL_SIZE,
            ha="left",
            va="top",
        )

        ax.set_ylabel("Probability density", fontsize=MEDIUM_SIZE)
        ax.set_title(title, fontsize=BIGGER_SIZE)
        ax.set_ylim(0, ymax_panel)
        ax.tick_params(axis="both", labelsize=SMALL_SIZE)

    axes[-1].set_xlabel(
        "log₁₀(SENSE / SOCA) social risk value", fontsize=MEDIUM_SIZE
    )
    axes[-1].set_xlim(XMIN, XMAX)

    # fig.suptitle removed — title goes in the manuscript figure caption

    fig.subplots_adjust(
        left=0.18,
        right=0.99,
        bottom=0.16,
        top=0.88,
        hspace=0.35,
    )

    if savepath:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved figure to: {savepath.resolve()}")
    else:
        plt.show()


# ============================================================
# MAIN ENTRY POINT
# ============================================================
def main(project_name=None, savepath=None):
    print("=== SENSE vs SOCA validation ===")
    print(f"Brightway project:    {project_name or PROJECT_NAME}")
    print(f"Technosphere DB:      {TECHNOSPHERE_DB}")
    print(f"Soca EcoSpold folder: {SPOLD_FOLDER}")
    print(f"Overview Excel:       {OVERVIEW_XLSX}")
    print(f"Cache dir:            {CACHE_DIR}\n")

    # Basic path checks for user friendliness
    if not SPOLD_FOLDER.exists():
        raise FileNotFoundError(
            f"SPOLD_FOLDER does not exist: {SPOLD_FOLDER}\n"
            "Please edit the USER CONFIG section at the top of the script."
        )
    if not OVERVIEW_XLSX.exists():
        raise FileNotFoundError(
            f"OVERVIEW_XLSX not found: {OVERVIEW_XLSX}\n"
            "Please edit the USER CONFIG section at the top of the script."
        )

    # 0) Brightway project / DB
    _project = project_name or PROJECT_NAME
    projects.set_current(_project)
    eco = Database(TECHNOSPHERE_DB)

    print("Using Brightway project:", projects.current)
    print("Using technosphere DB  :", eco.name, "\n")

    # 1) Parse EcoSpold once → metadata + XML social flows (SOCA)
    #    NOW CACHED:
    mapping_df, xml_df = load_or_parse_spold()

    # 2) Merge AO overview and match to Brightway
    spold_with_ao = build_ao_and_bw_match(mapping_df, eco)

    # 3) Social flows from Brightway (SENSE / PSILCA)
    bw_df = extract_bw_social_flows(eco, spold_with_ao)

    # 4) Indicator overview (optional sanity check)
    summarize_indicators(bw_df, xml_df)

    # 5) Activity-level coverage + Excel export
    summarize_activity_coverage_and_export(bw_df, xml_df, spold_with_ao, COVERAGE_EXCEL)

    # 6) Build compare_pos table with log10(SENSE / SOCA)
    compare_pos = build_compare(bw_df, xml_df)

    # 7) Final 3-panel geography plot
    plot_geography_panels(compare_pos, spold_with_ao, savepath=savepath)


main(project_name="sense_validation_arithmetic", savepath=Path("validation_arithmetic.svg"))
main(project_name="sense_validation_geometric", savepath=Path("validation_geometric.svg"))
